In [19]:
import pandas as pd
from pysus.online_data.SINAN import list_diseases, get_available_years
from pysus import get_version

In [18]:
info = list_diseases()

# INFL é a influenza pandemica.

In [20]:
get_version()

'1.0.1'

In [22]:

import pandas as pd

def catch_srag_parquet(url_parquet, uf='SP'):
    """
    Extrai dados do SIVEP-Gripe em formato Parquet (Camada Bronze -> Silver).
    
    Args:
        url_parquet (str): URL direta do arquivo .parquet no S3 do Ministério da Saúde.
        uf (str): Sigla do estado para filtragem.
        
    Returns:
        pd.DataFrame: DataFrame processado com os alvos identificados.
    """
    print(f"--- Iniciando Ingestão (Bronze) ---")
    print(f"Lendo arquivo Parquet de: {url_parquet.split('/')[-1]}")
    
    try:
        # Nota: Para rodar isso, você precisa das bibliotecas 'pyarrow' ou 'fastparquet' instaladas
        # O pandas detecta o formato automaticamente pela extensão
        df_raw = pd.read_parquet(url_parquet)
        
        print(f"Dados brutos carregados: {df_raw.shape[0]} linhas.")
        
        # --- CAMADA SILVER (Limpeza e Filtragem) ---
        print(f"Processando Camada Silver para {uf}...")
        
        # 1. Filtro por Estado (Notificação)
        df_silver = df_raw[df_raw['SG_UF_NOT'] == uf].copy()
        
        # 2. Seleção de Colunas Estratégicas
        cols = [
            'DT_NOTIFIC', 'SG_UF_NOT', 'ID_MUNICIP', 'NU_IDADE_N', 
            'CLASSI_FIN', 'PCR_VSR', 'PCR_INF_A', 'PCR_RINO', 
            'UTI', 'EVOLUCAO'
        ]
        
        cols_existentes = [c for c in cols if c in df_silver.columns]
        df_silver = df_silver[cols_existentes]
        
        def identificar_virus(row):
            if 'PCR_INF_A' in row and str(row['PCR_INF_A']) == '1': return 'Influenza A'
            if 'PCR_VSR' in row and str(row['PCR_VSR']) == '1': return 'VSR'
            if 'PCR_RINO' in row and str(row['PCR_RINO']) == '1': return 'Rinovírus'
            return 'Outros/Não Identificado'

        df_silver['VIRUS_ALVO'] = df_silver.apply(identificar_virus, axis=1)
        
        print(f"Processamento concluído. {df_silver.shape[0]} registros para {uf}.")
        return df_silver

    except Exception as e:
        print(f"Erro crítico no processamento do Parquet: {e}")
        return pd.DataFrame()


url_alvo = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2026/INFLUD26-23-03-2026.parquet"

df_sp = catch_srag_parquet(url_alvo, uf='SP')

if not df_sp.empty:
    print("\nResumo de Casos Identificados em SP:")
    print(df_sp['VIRUS_ALVO'].value_counts())

--- Iniciando Ingestão (Bronze) ---
Lendo arquivo Parquet de: INFLUD26-23-03-2026.parquet
Dados brutos carregados: 36546 linhas.
Processando Camada Silver para SP...
Processamento concluído. 7688 registros para SP.

Resumo de Casos Identificados em SP:
VIRUS_ALVO
Outros/Não Identificado    6810
Rinovírus                   767
VSR                         111
Name: count, dtype: int64
